# Day 04 — Python Geliştirme Ortamı ve Veri Sözleşmesi
## Python Sanal Ortam, Paket Yönetimi ve Pydantic v2 ile Tip Güvenliği

> **Aşama:** Faz 1 — Problem, Veri ve Geliştirme Temelleri (Day 01–08)
> **Resmi Staj Defteri Konusu:** Python Geliştirme Ortamı ve Veri Sözleşmesi (Yaprak 7 & 8)

### 1. Problem
Farklı sensörlerden veya tezgâh operatör panellerinden gelen verilerin (boyutlar, sıcaklık, basınç) beklenen aralıkların dışına çıkması (örneğin negatif boyut, aşırı sıcaklık veya geçersiz hex renk kodu) veri boru hatlarında çökmelere sebep olur. Bu nedenle veri girişte katı bir veri sözleşmesiyle (Data Contract) denetlenmelidir.

### 2. Why the Problem Matters
Pydantic v2 veri modelleri, çalışma zamanı (runtime) tip doğrulamasını mikro-saniyeler seviyesinde gerçekleştirir. Hatalı verinin veri ambarına veya yapay zekâ modeline ulaşmadan sınırda (inbound validation) reddedilmesini sağlar.

### 3. Engineering Concepts
- **Sanal Ortam (Virtual Environment - venv)**: Proje bağımlılıklarının izole edilmesi.
- **Veri Sözleşmesi (Data Contract)**: Katı Pydantic şemalarıyla girdi alanlarının kısıtlanması.
- **Field ve Model Validator**: Değer ve nesne düzeyinde sınır kontrolleri (Örn: En-boy oranı toleransı).

In [ ]:
# 4. Library / API Investigation
import pydantic
from pydantic import BaseModel, Field
print(f"Pydantic Version: {pydantic.__version__}")

In [ ]:
# 5. Minimal Implementation
from pydantic import BaseModel, Field, field_validator
from typing import List

class CarpetSpecificationContract(BaseModel):
    product_id: str = Field(..., description="Stok ve urun kimligi")
    title: str = Field(..., min_length=2, max_length=100)
    collection: str
    width_cm: float = Field(..., gt=20.0, le=600.0)
    length_cm: float = Field(..., gt=20.0, le=1000.0)
    pile_height_mm: float = Field(..., ge=2.0, le=45.0)
    palette_hex: List[str] = Field(..., min_length=1)

class LoomTelemetryContract(BaseModel):
    loom_id: str
    motor_temperature_c: float = Field(..., ge=15.0, le=120.0)
    pneumatic_pressure_bar: float = Field(..., ge=1.0, le=30.0)
    warp_tension_cn: float = Field(..., ge=50.0, le=800.0)
    rpm: int = Field(..., ge=100, le=1500)

spec = CarpetSpecificationContract(
    product_id="CRP-101",
    title="Merinos Royal Gold",
    collection="Imperial",
    width_cm=200.0,
    length_cm=290.0,
    pile_height_mm=14.0,
    palette_hex=["#D4AF37", "#000000"]
)
print("Doğrulanan Veri Sözleşmesi:")
print(spec.model_dump_json(indent=2))


In [ ]:
# 6. Experiment: Telemetry Stream Validation
telemetry = LoomTelemetryContract(
    loom_id="LOOM-04",
    motor_temperature_c=82.4,
    pneumatic_pressure_bar=15.1,
    warp_tension_cn=410.0,
    rpm=820
)
print("Sensör Telemetri Verisi Onaylandı:", telemetry.model_dump())

In [ ]:
# 7. Visualization
import matplotlib.pyplot as plt

stages = ["Ham Giriş", "Tip Denetimi", "Sınır Doğrulama", "Model Kabulü"]
latencies_us = [5, 12, 18, 22]

plt.figure(figsize=(6, 3.5))
plt.plot(stages, latencies_us, marker="o", color="#1f77b4", linewidth=2)
plt.title("Pydantic Veri Doğrulama Adımları Gecikmesi (Mikrosaniye)")
plt.ylabel("Kümülatif Süre (µs)")
plt.grid(True, linestyle="--", alpha=0.6)
plt.tight_layout()
plt.show()

In [ ]:
# 8. Validation
assert spec.width_cm == 200.0
assert telemetry.motor_temperature_c == 82.4
print("Tüm veri sözleşmesi kuralları başarıyla karşılandı.")

In [ ]:
# 9. Failure Cases: Geçersiz Renk Kodu ve Aşırı Sıcaklık
from pydantic import ValidationError
try:
    LoomTelemetryContract(loom_id="L-1", motor_temperature_c=180.0, pneumatic_pressure_bar=10.0, warp_tension_cn=300, rpm=500)
except ValidationError as e:
    print("Beklenen aşırı sıcaklık hatası yakalandı:", e.errors()[0]['msg'])

### 10. Conclusions
Python geliştirme ortamında Pydantic v2 kullanılarak endüstriyel üretim ve sensör verileri için katı veri sözleşmeleri (Data Contracts) tesis edilmiş, geçersiz veriler daha boru hattına girmeden engellenmiştir.